本文件用于论文各部分中间数据的生成 <br>
Part1-Cost：生成论文第一部分内容所需数据，生成的数据为Part1-cost.Rmd所需 <br>
Part2 - Multistate：生成论文第二部分内容所需数据，生成的数据为Part2-multistate.Rmd所需 <br>
Part3-survival outcomes：生成论文第三部分内容所需数据，生成的数据为Part3-Outcomes.Rmd所需 <br>

In [2]:
import pandas as pd
import numpy as np
import os
import re

## 医保类型转变统计
总共有 4732331 人 <br>
其中 4212565 人的医保类型没有改变过，占89.02%

In [ ]:
import pandas as pd
df = pd.read_parquet('File/hpconcat.parquet')

In [ ]:
groups = df.groupby('身份证号', as_index=False)['医疗付款方式']

In [ ]:
pay_change = (
    df.groupby('身份证号')['医疗付款方式']
      .nunique(dropna=True)
      .reset_index(name='付款方式种类数')
)

pay_change['付款方式种类数'].value_counts()

In [ ]:
summary = (
    pay_change['付款方式种类数']
    .value_counts()
    .sort_index()
    .reset_index()
)
summary.columns = ['付款方式种类数', '人数']
summary['占比'] = round((summary['人数'] / summary['人数'].sum())*100, 3)
print(summary)

In [ ]:
pay_change['是否改变'] = pay_change['付款方式种类数'].apply(
    lambda x: '始终不变' if x == 1 else '改变过'
)

change_summary = (
    pay_change['是否改变']
    .value_counts()
    .reset_index()
)
change_summary.columns = ['是否改变', '人数']
change_summary['占比'] = change_summary['人数'] / change_summary['人数'].sum()
print(change_summary)

In [ ]:
change_summary['人数'].sum()

# Part1 - Cost

1. 最开始数据           8174338
2. 清除费用缺失以及负数  8021254
3. 2019-2024           8000737

## main analysis

In [ ]:
df = pd.read_parquet(r"File/hpconcat.parquet")

In [ ]:
df['医疗付款方式'].value_counts()

In [ ]:
def correct_fee(x):
    if x['医疗付款方式'] == 'SP':
        value = x['住院费']
    else:
        value = x['自付金额']
    return value

df['自付金额'] = df[['医疗付款方式', '住院费', '自付金额']].apply(correct_fee, axis=1)

In [ ]:
print('住院费的缺失的住院记录有', ((df['住院费'].isna()) | (df['住院费'] <= 0)).sum(), '条')
print('自付金额的缺失的住院记录有', ((df['自付金额'].isna()) | (df['自付金额'] < 0)).sum(), '条')
print('费用缺失的住院记录有', (((df['住院费'].isna()) | (df['住院费'] <= 0)) | ((df['自付金额'].isna()) | (df['自付金额'] < 0))).sum(), '条')

In [ ]:
df = df[(df['住院费'].notna()) & (df['住院费'] > 0)]
df = df[df['自付金额'].notna() & (df['自付金额'] >= 0)]
print('删除费用缺失的患者后住院记录共有', df.shape, '条')

In [ ]:
df['身份证号'].unique()

In [ ]:
df['sum_fee'] = 0               # 总费用
df['sum_reimbursement'] = 0     # 总报销后费用
df['inpatient_times'] = 0       # 总住院次数

In [ ]:
df.sort_values(by=['身份证号', '入院时间'], inplace=True)
df['sum_fee'] = df.groupby('身份证号')['住院费'].cumsum()
df['sum_reimbursement'] = df.groupby('身份证号')['自付金额'].cumsum()
df['inpatient_times'] = df.groupby('身份证号')['姓名'].cumcount() + 1

In [ ]:
# 并发症确定

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df['hypertension'] = df['主要诊断编码'].isin(hp).astype(int)
df['heart'] = df['主要诊断编码'].isin(heart).astype(int)
df['brain'] = df['主要诊断编码'].isin(brain).astype(int)
df['kidney'] = df['主要诊断编码'].isin(kidney).astype(int)

In [ ]:
cols_basic = ['姓名', '身份证号', '年龄', '性别', '入院时间', '医疗付款方式', '市', '区县']
cols_sum = ['sum_fee', 'sum_reimbursement', 'inpatient_times']
cols_disease = ['hypertension', 'heart', 'brain', 'kidney']

# 1. 排序
df_sorted = df.sort_values(['身份证号', '入院时间'])

# 2. 获取第一次入院的基本信息
first_records = (
    df_sorted.groupby('身份证号', as_index=False)
    .first()[cols_basic]
)

# 3. 按身份证号聚合：费用求和、入院次数求和，疾病取最大值
agg_records = (
    df_sorted.groupby('身份证号', as_index=False)
    .agg({
        'sum_fee': 'max',
        'sum_reimbursement': 'max',
        'inpatient_times': 'max',
        'hypertension': 'max',
        'heart': 'max',
        'brain': 'max',
        'kidney': 'max',
    })
)

# 4. 合并首次基本信息 + 聚合结果
result = first_records.merge(agg_records, on='身份证号')

In [ ]:
result.to_parquet('File/Part Data/part1-main.parquet')

## sensitivity analysis

### only hp
初始没有并发症

In [ ]:
df = pd.read_parquet(r"File/hpconcat.parquet")

In [ ]:
def correct_fee(x):
    if x['医疗付款方式'] == 'SP':
        value = x['住院费']
    else:
        value = x['自付金额']
    return value

df['自付金额'] = df[['医疗付款方式', '住院费', '自付金额']].apply(correct_fee, axis=1)

In [ ]:
df = df[(df['住院费'].notna()) & (df['住院费'] > 0)]
df = df[df['自付金额'].notna() & (df['自付金额'] >= 0)]
print('删除费用缺失的患者后住院记录共有', df.shape, '条')

In [ ]:
df['sum_fee'] = 0               # 总费用
df['sum_reimbursement'] = 0     # 总报销后费用
df['inpatient_times'] = 0       # 总住院次数

In [ ]:
df.sort_values(by=['身份证号', '入院时间'], inplace=True)
df['sum_fee'] = df.groupby('身份证号')['住院费'].cumsum()
df['sum_reimbursement'] = df.groupby('身份证号')['自付金额'].cumsum()
df['inpatient_times'] = df.groupby('身份证号')['姓名'].cumcount() + 1

In [ ]:
# 并发症确定

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df['hypertension'] = df['主要诊断编码'].isin(hp).astype(int)
df['heart'] = df['主要诊断编码'].isin(heart).astype(int)
df['brain'] = df['主要诊断编码'].isin(brain).astype(int)
df['kidney'] = df['主要诊断编码'].isin(kidney).astype(int)

In [ ]:
# 敏感性分析，首次入院没有并发症的患者
cols_basic = ['姓名', '身份证号', '年龄', '性别', '入院时间', '医疗付款方式', '市', '区县']
cols_sum = ['sum_fee', 'sum_reimbursement', 'inpatient_times']
cols_disease = ['hypertension', 'heart', 'brain', 'kidney']

# 1. 排序
df_sorted = df.sort_values(['身份证号', '入院时间'])

# 2. 获取第一次入院的基本信息 + 首次疾病情况
first_records = (
    df_sorted.groupby('身份证号', as_index=False)
    .first()[cols_basic + cols_disease]
)

# 3. 只保留首次疾病全为0的患者
mask = (first_records[cols_disease].sum(axis=1) == 0)
first_records = first_records.loc[mask]
first_records = first_records[cols_basic]

# 4. 聚合：费用/次数取 max，疾病取 max
agg_records = (
    df_sorted.groupby('身份证号', as_index=False)
    .agg({
        'sum_fee': 'max',
        'sum_reimbursement': 'max',
        'inpatient_times': 'max',
        'hypertension': 'max',
        'heart': 'max',
        'brain': 'max',
        'kidney': 'max',
    })
)

# 5. 合并，确保只保留符合条件的身份证号
result = first_records.merge(agg_records, on='身份证号')

In [ ]:
result.to_parquet('File/Part Data/part1-sensitivity-onlyhp.parquet')

### payment unchanged

In [ ]:
df = pd.read_parquet(r"File/hpconcat.parquet")

In [ ]:
def correct_fee(x):
    if x['医疗付款方式'] == 'SP':
        value = x['住院费']
    else:
        value = x['自付金额']
    return value

df['自付金额'] = df[['医疗付款方式', '住院费', '自付金额']].apply(correct_fee, axis=1)

In [ ]:
df = df[(df['住院费'].notna()) & (df['住院费'] > 0)]
df = df[df['自付金额'].notna() & (df['自付金额'] >= 0)]
print('删除费用缺失的患者后住院记录共有', df.shape, '条')

In [ ]:
unchanged_id = pay_change[pay_change['付款方式种类数'] == 1]['身份证号']
df_unchanged = df[df['身份证号'].isin(unchanged_id)].copy()

print('没有改变过的医疗支付方式的记录有', df_unchanged.shape[0], '条', '\n' ,'共有', len(df_unchanged['身份证号'].unique()),'人')

In [ ]:
df_unchanged['sum_fee'] = 0             # 总费用
df_unchanged['sum_reimbursement'] = 0   # 总报销
df_unchanged['inpatient_times'] = 0     # 住院次数
df_unchanged['comorbidity_times'] = 0   # 并发症次数

In [ ]:
df_unchanged.sort_values(by=['身份证号', '入院时间'], inplace=True)
df_unchanged['sum_fee'] = df_unchanged.groupby('身份证号')['住院费'].cumsum()
df_unchanged['sum_reimbursement'] = df_unchanged.groupby('身份证号')['自付金额'].cumsum()
df_unchanged['inpatient_times'] = df_unchanged.groupby('身份证号')['姓名'].cumcount() + 1

In [ ]:

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
# 外周      I70-I79
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df_unchanged['hypertension'] = df_unchanged['主要诊断编码'].isin(hp).astype(int)
df_unchanged['heart'] = df_unchanged['主要诊断编码'].isin(heart).astype(int)
df_unchanged['brain'] = df_unchanged['主要诊断编码'].isin(brain).astype(int)
df_unchanged['kidney'] = df_unchanged['主要诊断编码'].isin(kidney).astype(int)

In [ ]:
cols_basic = ['姓名', '身份证号', '年龄', '性别', '入院时间', '医疗付款方式', '市', '区县']
cols_sum = ['sum_fee', 'sum_reimbursement', 'inpatient_times']
cols_disease = ['hypertension', 'heart', 'brain', 'kidney']

# 1. 排序
df_unchanged_sorted = df_unchanged.sort_values(['身份证号', '入院时间'])

# 2. 获取第一次入院的基本信息
first_records = (
    df_unchanged_sorted.groupby('身份证号', as_index=False)
    .first()[cols_basic]
)

# 3. 按身份证号聚合：费用求和、入院次数求和，疾病取最大值
agg_records = (
    df_unchanged_sorted.groupby('身份证号', as_index=False)
    .agg({
        'sum_fee': 'max',
        'sum_reimbursement': 'max',
        'inpatient_times': 'max',
        'hypertension': 'max',
        'heart': 'max',
        'brain': 'max',
        'kidney': 'max',
    })
)

# 4. 合并首次基本信息 + 聚合结果
result = first_records.merge(agg_records, on='身份证号')

In [ ]:
result.to_parquet('File/Part Data/part1-sensitivity-unchanged.parquet')

### Covid-19

In [ ]:
df = pd.read_parquet(r"File/hpconcat.parquet")

In [ ]:
def correct_fee(x):
    if x['医疗付款方式'] == 'SP':
        value = x['住院费']
    else:
        value = x['自付金额']
    return value

df['自付金额'] = df[['医疗付款方式', '住院费', '自付金额']].apply(correct_fee, axis=1)

In [ ]:
df = df[(df['住院费'].notna()) & (df['住院费'] > 0)]
df = df[df['自付金额'].notna() & (df['自付金额'] >= 0)]
print('删除费用缺失的患者后住院记录共有', df.shape, '条')

In [ ]:
df['sum_fee'] = 0               # 总费用
df['sum_reimbursement'] = 0     # 总报销后费用
df['inpatient_times'] = 0       # 总住院次数

In [ ]:
df.sort_values(by=['身份证号', '入院时间'], inplace=True)
df['sum_fee'] = df.groupby('身份证号')['住院费'].cumsum()
df['sum_reimbursement'] = df.groupby('身份证号')['自付金额'].cumsum()
df['inpatient_times'] = df.groupby('身份证号')['姓名'].cumcount() + 1

In [ ]:
# 并发症确定

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df['hypertension'] = df['主要诊断编码'].isin(hp).astype(int)
df['heart'] = df['主要诊断编码'].isin(heart).astype(int)
df['brain'] = df['主要诊断编码'].isin(brain).astype(int)
df['kidney'] = df['主要诊断编码'].isin(kidney).astype(int)

In [ ]:
cols_basic = ['姓名', '身份证号', '年龄', '性别', '入院时间', '医疗付款方式', '市', '区县']
cols_sum = ['sum_fee', 'sum_reimbursement', 'inpatient_times']
cols_disease = ['hypertension', 'heart', 'brain', 'kidney']

# 1. 排序
df_sorted = df.sort_values(['身份证号', '入院时间'])

df_before = df_sorted[df_sorted['入院时间'].dt.year <= 2022]
df_after = df_sorted[df_sorted['入院时间'].dt.year > 2022]

# 2. 获取第一次入院的基本信息
first_records = (
    df_sorted.groupby('身份证号', as_index=False)
    .first()[cols_basic]
)

# 3. 按身份证号聚合：费用求和、入院次数求和，疾病取最大值
agg_records_before = (
    df_before.groupby('身份证号', as_index=False)
    .agg({
        'sum_fee': 'max',
        'sum_reimbursement': 'max',
        'inpatient_times': 'max',
        'hypertension': 'max',
        'heart': 'max',
        'brain': 'max',
        'kidney': 'max',
    })
)

agg_records_after = (
    df_after.groupby('身份证号', as_index=False)
    .agg({
        'sum_fee': 'max',
        'sum_reimbursement': 'max',
        'inpatient_times': 'max',
        'hypertension': 'max',
        'heart': 'max',
        'brain': 'max',
        'kidney': 'max',
    })
)

# 4. 合并首次基本信息 + 聚合结果
result_before = first_records.merge(agg_records_before, on='身份证号')
result_after = first_records.merge(agg_records_after, on='身份证号')

In [ ]:
result_before.to_parquet('File/Part Data/part1-sensitivity-covid-before.parquet')
result_after.to_parquet('File/Part Data/part1-sensitivity-covid-after.parquet')

# Part2 - Multistate

## main

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ========================
# 1. 读取数据
# ========================
df = pd.read_parquet(r"File/hpconcat.parquet").copy()
df = df.reset_index(drop=True)

df["入院时间"] = pd.to_datetime(df["入院时间"], errors="coerce")
df = df.dropna(subset=["身份证号", "入院时间"]).copy()

# ========================
# 2. ICD 编码定义
# ========================
icd_cols = [
    "主要诊断编码", "疾病编码2", "疾病编码3",
    "疾病编码4", "疾病编码5", "疾病编码6", "疾病编码7",
    "疾病编码8", "疾病编码9", "疾病编码10", "疾病编码11",
    "疾病编码12", "疾病编码13", "疾病编码14", "疾病编码15"
]

heart_codes = [
    "I20", "I21", "I22", "I23", "I24", "I25",
    "I26", "I27", "I28",
    "I30", "I31", "I32", "I33", "I34", "I35", "I36", "I37", "I38", "I39",
    "I40", "I41", "I42", "I43", "I44", "I45", "I46", "I47", "I48", "I49",
    "I50", "I51", "I52"
]
brain_codes = ["I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68", "I69"]
kidney_codes = ["N04", "N18"]

heart_set = set(heart_codes)
brain_set = set(brain_codes)
kidney_set = set(kidney_codes)

In [ ]:
# ========================
# 3. 排序并提取首次住院
# ========================
df = df.sort_values(["身份证号", "入院时间"]).copy()
df_first = df.groupby("身份证号", as_index=False).first().copy()

# ========================
# 4. 主分析：首次住院必须为 state1
#    基线用全部诊断列判断是否已有目标并发症
# ========================
first_heart = df_first[icd_cols].isin(heart_set).any(axis=1)
first_brain = df_first[icd_cols].isin(brain_set).any(axis=1)
first_kidney = df_first[icd_cols].isin(kidney_set).any(axis=1)

bad_ids = df_first.loc[first_heart | first_brain | first_kidney, "身份证号"].unique()
df_filter = df[~df["身份证号"].isin(bad_ids)].copy()

# ========================
# 5. 随访阶段：用全部诊断列判断并发症
#    允许观测上出现 1 -> 5
# ========================
df_filter["heart_follow"] = df_filter[icd_cols].isin(heart_set).any(axis=1).astype(int)
df_filter["brain_follow"] = df_filter[icd_cols].isin(brain_set).any(axis=1).astype(int)
df_filter["kidney_follow"] = df_filter[icd_cols].isin(kidney_set).any(axis=1).astype(int)

In [ ]:
df_filter.shape

In [ ]:
df_filter['身份证号'].unique()

In [ ]:
# ========================
# 6. 构造 df_states
#    - 基线固定从 state1 开始
#    - 后续按累计病史更新
#    - 首次进入 state5 后立即停止记录
#    - 末次状态不是5时才补行政截止点
# ========================
study_end = pd.Timestamp("2024-12-31")
all_results = []

groups = df_filter.groupby("身份证号", sort=False)

for pid, group in tqdm(groups, desc="Building df_states"):
    group = group.sort_values(["入院时间"]).copy()
    baseline_time = group.iloc[0]["入院时间"]

    heart_hist = 0
    brain_hist = 0
    kidney_hist = 0

    patient_rows = []

    for i, (_, row) in enumerate(group.iterrows()):
        # 首次住院固定视为 state1，不更新基线并发症历史
        if i > 0:
            if int(row["heart_follow"]) == 1:
                heart_hist = 1
            if int(row["brain_follow"]) == 1:
                brain_hist = 1
            if int(row["kidney_follow"]) == 1:
                kidney_hist = 1

        n_comp = heart_hist + brain_hist + kidney_hist

        if n_comp == 0:
            state = 1
        elif n_comp == 1:
            if heart_hist == 1:
                state = 2
            elif brain_hist == 1:
                state = 3
            elif kidney_hist == 1:
                state = 4
            else:
                raise ValueError(f"{pid} 的单并发症状态无法识别")
        else:
            state = 5

        time_days = (row["入院时间"] - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": row["身份证号"],
            "payment": row["医疗付款方式"],
            "age": row["年龄"],
            "gender": row["性别"],
            "state": state,
            "time": row["入院时间"],
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 0
        })

        # 一旦首次进入 state5，立即停止后续记录
        if state == 5:
            break

    # 若末次状态不是5，且末次时间早于研究截止，则补行政截止点
    last_state = patient_rows[-1]["state"]
    last_time = patient_rows[-1]["time"]

    if (last_state != 5) and (pd.notna(last_time)) and (last_time < study_end):
        time_days = (study_end - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": patient_rows[-1]["身份证号"],
            "payment": patient_rows[-1]["payment"],
            "age": patient_rows[-1]["age"],
            "gender": patient_rows[-1]["gender"],
            "state": last_state,
            "time": study_end,
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 1
        })

    all_results.extend(patient_rows)

df_states = pd.DataFrame(all_results)

In [ ]:
# ========================
# 7. 排序与检查
# ========================
df_states = df_states.sort_values(["身份证号", "time"]).reset_index(drop=True)
df_states["time_years_rounded"] = df_states["time_years"].round(4)

print("=" * 60)
print("状态分布：")
print(df_states["state"].value_counts().sort_index())

print("=" * 60)
print("行政补点数量：")
print(df_states["is_admin_censor"].sum())

print("=" * 60)
print("每位患者记录数分布（前10项）：")
print(df_states.groupby("身份证号").size().value_counts().sort_index().head(10))

# 相邻观测转移表
df_check = df_states.sort_values(["身份证号", "time"]).copy()
df_check["next_state"] = df_check.groupby("身份证号")["state"].shift(-1)

trans_table = (
    df_check.dropna(subset=["next_state"])
    .groupby(["state", "next_state"])
    .size()
    .reset_index(name="n")
    .sort_values(["state", "next_state"])
)

print("=" * 60)
print("相邻观测转移表：")
print(trans_table)

In [ ]:
# ========================
# 8. 导出
# ========================
out_path = r"File/Part Data/part2-main.parquet"
df_states.to_parquet(out_path, index=False)
print(f"数据已保存至 {out_path}")

## sensitivity analysis

### primary icd
后续住院按照主要诊断

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ========================
# 1. 读取数据
# ========================
df = pd.read_parquet(r"File/hpconcat.parquet").copy()
df = df.reset_index(drop=True)

# 如果这里时间还不是 datetime，就保留；若之前已处理可删除
df["入院时间"] = pd.to_datetime(df["入院时间"], errors="coerce")
df = df.dropna(subset=["身份证号", "入院时间"]).copy()

# ========================
# 2. ICD 编码定义
# ========================
icd_cols = [
    "主要诊断编码", "疾病编码2", "疾病编码3",
    "疾病编码4", "疾病编码5", "疾病编码6", "疾病编码7",
    "疾病编码8", "疾病编码9", "疾病编码10", "疾病编码11",
    "疾病编码12", "疾病编码13", "疾病编码14", "疾病编码15"
]

heart_codes = [
    "I20", "I21", "I22", "I23", "I24", "I25",
    "I26", "I27", "I28",
    "I30", "I31", "I32", "I33", "I34", "I35", "I36", "I37", "I38", "I39",
    "I40", "I41", "I42", "I43", "I44", "I45", "I46", "I47", "I48", "I49",
    "I50", "I51", "I52"
]
brain_codes = ["I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68", "I69"]
kidney_codes = ["N04", "N18"]

heart_set = set(heart_codes)
brain_set = set(brain_codes)
kidney_set = set(kidney_codes)

In [ ]:
# ========================
# 3. 排序并提取首次住院
# ========================
df = df.sort_values(["身份证号", "入院时间"]).copy()
df_first = df.groupby("身份证号", as_index=False).first().copy()

# ========================
# 4. 主分析：首次住院必须为 state1
#    基线用全部诊断列判断是否已有目标并发症
# ========================
first_heart = df_first[icd_cols].isin(heart_set).any(axis=1)
first_brain = df_first[icd_cols].isin(brain_set).any(axis=1)
first_kidney = df_first[icd_cols].isin(kidney_set).any(axis=1)

bad_ids = df_first.loc[first_heart | first_brain | first_kidney, "身份证号"].unique()
df_filter = df[~df["身份证号"].isin(bad_ids)].copy()

# ========================
# 5. 随访阶段：用全部诊断列判断并发症
#    允许观测上出现 1 -> 5
# 敏感性分析：后续诊断使用主要诊断判定
# ========================
df_filter["heart_follow"] = df_filter['主要诊断编码'].isin(heart_set).astype(int)
df_filter["brain_follow"] = df_filter['主要诊断编码'].isin(brain_set).astype(int)
df_filter["kidney_follow"] = df_filter['主要诊断编码'].isin(kidney_set).astype(int)

In [ ]:
# ========================
# 6. 构造 df_states
#    - 基线固定从 state1 开始
#    - 后续按累计病史更新
#    - 首次进入 state5 后立即停止记录
#    - 末次状态不是5时才补行政截止点
# ========================
study_end = pd.Timestamp("2024-12-31")
all_results = []

groups = df_filter.groupby("身份证号", sort=False)

for pid, group in tqdm(groups, desc="Building df_states"):
    group = group.sort_values(["入院时间"]).copy()
    baseline_time = group.iloc[0]["入院时间"]

    heart_hist = 0
    brain_hist = 0
    kidney_hist = 0

    patient_rows = []

    for i, (_, row) in enumerate(group.iterrows()):
        # 首次住院固定视为 state1，不更新基线并发症历史
        if i > 0:
            if int(row["heart_follow"]) == 1:
                heart_hist = 1
            if int(row["brain_follow"]) == 1:
                brain_hist = 1
            if int(row["kidney_follow"]) == 1:
                kidney_hist = 1

        n_comp = heart_hist + brain_hist + kidney_hist

        if n_comp == 0:
            state = 1
        elif n_comp == 1:
            if heart_hist == 1:
                state = 2
            elif brain_hist == 1:
                state = 3
            elif kidney_hist == 1:
                state = 4
            else:
                raise ValueError(f"{pid} 的单并发症状态无法识别")
        else:
            state = 5

        time_days = (row["入院时间"] - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": row["身份证号"],
            "payment": row["医疗付款方式"],
            "age": row["年龄"],
            "gender": row["性别"],
            "state": state,
            "time": row["入院时间"],
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 0
        })

        # 一旦首次进入 state5，立即停止后续记录
        if state == 5:
            break

    # 若末次状态不是5，且末次时间早于研究截止，则补行政截止点
    last_state = patient_rows[-1]["state"]
    last_time = patient_rows[-1]["time"]

    if (last_state != 5) and (pd.notna(last_time)) and (last_time < study_end):
        time_days = (study_end - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": patient_rows[-1]["身份证号"],
            "payment": patient_rows[-1]["payment"],
            "age": patient_rows[-1]["age"],
            "gender": patient_rows[-1]["gender"],
            "state": last_state,
            "time": study_end,
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 1
        })

    all_results.extend(patient_rows)

df_states = pd.DataFrame(all_results)

In [ ]:
# ========================
# 7. 排序与检查
# ========================
df_states = df_states.sort_values(["身份证号", "time"]).reset_index(drop=True)
df_states["time_years_rounded"] = df_states["time_years"].round(4)

print("=" * 60)
print("状态分布：")
print(df_states["state"].value_counts().sort_index())

print("=" * 60)
print("行政补点数量：")
print(df_states["is_admin_censor"].sum())

print("=" * 60)
print("每位患者记录数分布（前10项）：")
print(df_states.groupby("身份证号").size().value_counts().sort_index().head(10))

# 相邻观测转移表
df_check = df_states.sort_values(["身份证号", "time"]).copy()
df_check["next_state"] = df_check.groupby("身份证号")["state"].shift(-1)

trans_table = (
    df_check.dropna(subset=["next_state"])
    .groupby(["state", "next_state"])
    .size()
    .reset_index(name="n")
    .sort_values(["state", "next_state"])
)

print("=" * 60)
print("相邻观测转移表：")
print(trans_table)

In [ ]:
# ========================
# 8. 导出
# ========================
out_path = r"File/Part Data/part2-sensitivity-primaryicds.parquet"
df_states.to_parquet(out_path, index=False)
print(f"数据已保存至 {out_path}")

### start at any state
初始状态可以为state1, state2, state2, state3 <br>
不再限制初始状态为state1

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ========================
# 读取数据
# ========================
df = pd.read_parquet(r"File/hpconcat.parquet").copy()
df = df.reset_index(drop=True)

# ========================
# ICD 编码定义
# ========================
icd_cols = [
    "主要诊断编码", "疾病编码2", "疾病编码3",
    "疾病编码4", "疾病编码5", "疾病编码6", "疾病编码7",
    "疾病编码8", "疾病编码9", "疾病编码10", "疾病编码11",
    "疾病编码12", "疾病编码13", "疾病编码14", "疾病编码15"
]

heart_codes = [
    "I20", "I21", "I22", "I23", "I24", "I25",
    "I26", "I27", "I28",
    "I30", "I31", "I32", "I33", "I34", "I35", "I36", "I37", "I38", "I39",
    "I40", "I41", "I42", "I43", "I44", "I45", "I46", "I47", "I48", "I49",
    "I50", "I51", "I52"
]
brain_codes = ["I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68", "I69"]
kidney_codes = ["N04", "N18"]

heart_set = set(heart_codes)
brain_set = set(brain_codes)
kidney_set = set(kidney_codes)

In [ ]:
# ========================
# 排序并提取首次住院
# ========================
df = df.sort_values(["身份证号", "入院时间"]).copy()
df_first = df.groupby("身份证号", as_index=False).first().copy()

# 判断首次住院的并发症
first_heart = df_first[icd_cols].isin(heart_set).any(axis=1).astype(int)
first_brain = df_first[icd_cols].isin(brain_set).any(axis=1).astype(int)
first_kidney = df_first[icd_cols].isin(kidney_set).any(axis=1).astype(int)

df_first["heart_base"] = first_heart
df_first["brain_base"] = first_brain
df_first["kidney_base"] = first_kidney
df_first["n_comp_base"] = df_first[["heart_base", "brain_base", "kidney_base"]].sum(axis=1)

# 获取首次住院的状态
def get_state_from_history(heart_hist, brain_hist, kidney_hist):
    n_comp = int(heart_hist) + int(brain_hist) + int(kidney_hist)
    if n_comp == 0:
        return 1
    elif n_comp == 1:
        if heart_hist == 1:
            return 2
        elif brain_hist == 1:
            return 3
        elif kidney_hist == 1:
            return 4
        else:
            raise ValueError("单并发症状态无法识别")
    else:
        return 5

df_first["state_base"] = df_first.apply(
    lambda row: get_state_from_history(row["heart_base"], row["brain_base"], row["kidney_base"]),
    axis=1
)

# 仅保留首次住院状态映射表
base_info = df_first[[
    "身份证号", "heart_base", "brain_base", "kidney_base", "state_base"
]].copy()

# ========================
# 随访阶段并发症判断（全部诊断列）
# ========================
df["heart_follow"] = df[icd_cols].isin(heart_set).any(axis=1).astype(int)
df["brain_follow"] = df[icd_cols].isin(brain_set).any(axis=1).astype(int)
df["kidney_follow"] = df[icd_cols].isin(kidney_set).any(axis=1).astype(int)

# 合并基线状态信息
df = df.merge(base_info, on="身份证号", how="left")

In [ ]:
# ========================
# 构造 df_states
#    - 基线不再要求从 state1 开始
#    - 后续按累计病史更新
#    - 末次状态不是 5 时才补行政截止点
# ========================
study_end = pd.Timestamp("2024-12-31")
all_results = []

groups = df.groupby("身份证号", sort=False)

for pid, group in tqdm(groups, desc="Building df_states_sens_anybasestate"):
    group = group.sort_values(["入院时间"]).copy()
    baseline_time = group.iloc[0]["入院时间"]

    # 读取基线累计病史
    heart_hist = int(group.iloc[0]["heart_base"])
    brain_hist = int(group.iloc[0]["brain_base"])
    kidney_hist = int(group.iloc[0]["kidney_base"])

    patient_rows = []

    for i, (_, row) in enumerate(group.iterrows()):
        # 第一次住院：直接使用基线状态，不更新基线并发症历史
        if i == 0:
            state = int(row["state_base"])
        else:
            # 后续住院按累计病史更新
            if int(row["heart_follow"]) == 1:
                heart_hist = 1
            if int(row["brain_follow"]) == 1:
                brain_hist = 1
            if int(row["kidney_follow"]) == 1:
                kidney_hist = 1

        n_comp = heart_hist + brain_hist + kidney_hist

        if n_comp == 0:
            state = 1
        elif n_comp == 1:
            if heart_hist == 1:
                state = 2
            elif brain_hist == 1:
                state = 3
            elif kidney_hist == 1:
                state = 4
            else:
                state = 1  # 再次检查确认
        else:
            state = 5

        time_days = (row["入院时间"] - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": row["身份证号"],
            "payment": row["医疗付款方式"],
            "age": row["年龄"],
            "gender": row["性别"],
            "state": state,
            "time": row["入院时间"],
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 0
        })

        # 一旦首次进入 state5，立即停止后续记录
        if state == 5:
            break

    # 若末次状态不是5，且末次时间早于研究截止，则补行政截止点
    last_state = patient_rows[-1]["state"]
    last_time = patient_rows[-1]["time"]

    if (last_state != 5) and (pd.notna(last_time)) and (last_time < study_end):
        time_days = (study_end - baseline_time).days
        time_years = time_days / 365.25

        patient_rows.append({
            "身份证号": patient_rows[-1]["身份证号"],
            "payment": patient_rows[-1]["payment"],
            "age": patient_rows[-1]["age"],
            "gender": patient_rows[-1]["gender"],
            "state": last_state,
            "time": study_end,
            "time_days": time_days,
            "time_years": time_years,
            "is_admin_censor": 1
        })

    all_results.extend(patient_rows)

df_states_sens_anybasestate = pd.DataFrame(all_results)

In [ ]:
# ========================
# 7. 排序与检查s
# ========================
df_states_sens_anybasestate = df_states_sens_anybasestate.sort_values(["身份证号", "time"]).reset_index(drop=True)
df_states_sens_anybasestate["time_years_rounded"] = df_states_sens_anybasestate["time_years"].round(4)

print("=" * 60)
print("状态分布：")
print(df_states_sens_anybasestate["state"].value_counts().sort_index())

print("=" * 60)
print("行政补点数量：")
print(df_states_sens_anybasestate["is_admin_censor"].sum())

print("=" * 60)
print("每位患者记录数分布（前10项）：")
print(df_states_sens_anybasestate.groupby("身份证号").size().value_counts().sort_index().head(10))

# 相邻观测转移表
df_check = df_states_sens_anybasestate.sort_values(["身份证号", "time"]).copy()
df_check["next_state"] = df_check.groupby("身份证号")["state"].shift(-1)

trans_table = (
    df_check.dropna(subset=["next_state"])
    .groupby(["state", "next_state"])
    .size()
    .reset_index(name="n")
    .sort_values(["state", "next_state"])
)

print("=" * 60)
print("相邻观测转移表：")
print(trans_table)

In [ ]:

# ========================
# 8. 导出
# ========================
# out_path = r"File/Part Data/part2-sensitivity-anystate.parquet"
# df_states_sens_anybasestate.to_parquet(out_path, index=False)
# print(f"数据已保存至 {out_path}")

# Part3-survival outcomes

## Mian

In [ ]:
import msoffcrypto
import io
import os
import pandas as pd

In [ ]:
icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

In [ ]:
county_list = [
    '范县', '方城县', '获嘉县',
    '郏县', '卢氏县', '杞县',
    '汤阴县', '武陟县', '夏邑县',
    '叶县'
]

df = pd.read_parquet(r"File\hpconcat.parquet")
df = df[df['区县'].isin(county_list)]

df = df.sort_values(by=['身份证号', '出生日期'], ascending=[True, True])
df = df.drop_duplicates(subset=['身份证号', '出生日期'], keep='first')
df['身份证号'] = df['身份证号'].apply(lambda x: x[-4:] if pd.notna(x) else x)
df = df.loc[:, ['姓名', '身份证号', '性别', '医疗付款方式','年龄', '区县', '入院时间', '出院时间', '住院费', '自付金额'] + icd_cols]

In [ ]:
df.shape

In [ ]:
# 合并死亡数据
outcomes_list = []   
death_path = r"E:\DataBase\全省his系统\死亡数据"
password = os.environ.get('Death_code')

for i in county_list:
    # print(i)
    outcome_path = os.path.join(death_path, i, '死亡状态登记报告.xlsx')
    with open(outcome_path, 'rb') as f:
        office_file = msoffcrypto.OfficeFile(f)
        office_file.load_key(password=password)
        
        decrypted = io.BytesIO()
        office_file.decrypt(decrypted)  # 解密到内存

    death_df = pd.read_excel(decrypted, sheet_name=0)
    death_df['身份证号'] = death_df['身份证件号码'].apply(lambda x: x[-4:] if pd.notna(x) else x)

    outcomes_list.append(death_df.loc[:, ['姓名','身份证号', '死亡日期', '根本死因']].copy())
    
    death_df = pd.concat(outcomes_list)

In [ ]:
# 合并
df = pd.merge(df, death_df, on=['姓名','身份证号'], how='left')

In [ ]:
# 清洗日期格式
def clean_date_column(df, colname):
    """
    清洗日期列，统一格式为 YYYY-MM-DD
    - 支持斜杠/横杠
    - 自动补零（单个数字前补0）
    - 去除时分秒
    - 解析失败的值转换为 NaT
    """
    # 1. 转为字符串并清理
    df[colname] = (
        df[colname]
        .astype(str)
        .str.strip()
        .str.replace('/', '-', regex=False)                     # 统一分隔符
        .str.replace(r'(\D)(\d)(?=\D)', r'\g<1>0\2', regex=True)  # 单位数补零
        .str.slice(0, 10)                                       # 保留前10位 (YYYY-MM-DD)
    )

    # 2. 转为日期
    df[colname] = pd.to_datetime(df[colname], errors='coerce').dt.normalize()


    return df  

df = clean_date_column(df, '入院时间')
df = clean_date_column(df, '死亡日期')

In [ ]:
# 规范日期格式
def preprocess(df, start_time, end_time):
    """
    处理数据
    """

    # 入院时间
    df = df[(df['入院时间'] < pd.to_datetime(end_time)) & (df['入院时间'] > pd.to_datetime(start_time))]
    # 死亡日期
    df = df[(df['死亡日期'] < pd.to_datetime(end_time)) | pd.isna(df['死亡日期'])]
    # 入院时间-死亡日期
    df = df[(df['死亡日期'] > df['入院时间']) | pd.isna(df['死亡日期'])]

    # 去重
    df = df.drop_duplicates(subset=['姓名', '身份证号'], keep='first')

    # 重置索引
    df = df.reset_index(drop=True)

    return df

df = preprocess(df, start_time='2019-01-01', end_time='2024-12-31')

In [ ]:
icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

df['heart'] = df[icd_cols].isin(heart).any(axis=1).astype(int)
df['brain'] = df[icd_cols].isin(brain).any(axis=1).astype(int)
df['kidney'] = df[icd_cols].isin(kidney).any(axis=1).astype(int)

In [ ]:
# 生成follow_time和event
def creat_km_data(df, end_time, regex = False, time_col='follow_time', event_col='event'):
    """
    生成生存数据
    """
    import re

    # 随访时间
    end_time = pd.to_datetime(end_time)
    df['结束时间'] = df['死亡日期'].fillna(end_time)
    df[time_col] = (df['结束时间'] - df['入院时间']).dt.days / 30

    # 随访结局
    df[event_col] = 0
    if regex:
        df['status'] = 0

        mask = df['根本死因'].str.contains(regex, na=False, regex=True)
        df.loc[mask, event_col] = 1

        df.loc[mask, 'status'] = 1
        df.loc[(~mask) & (pd.notna(df['死亡日期'])), 'status'] = 2
    else:
        df.loc[pd.notna(df['死亡日期']), event_col] = 1

    return df

df = creat_km_data(df, end_time='2024-12-31', time_col='allcause_follow_time', event_col='allcause_event')
df = creat_km_data(df, end_time='2024-12-31', regex='.*心.*|.*脑.*',time_col='spe_follow_time', event_col='spe_event')

In [ ]:
# 保存数据
# df.to_parquet('File/Part Data/part3-main.parquet')
# df = pd.read_parquet('File/Part Data/part3-main.parquet')

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# 0 生存曲线
data_1 = df[df['医疗付款方式'] == 'URRBMI']
kmf_group1 = KaplanMeierFitter()
kmf_group1.fit(
    durations=data_1['allcause_follow_time'], 
    event_observed=data_1['allcause_event'], 
    label='URBMI'
)

# 1 生存曲线
data_2 = df[df['医疗付款方式'] == 'UEBMI']
kmf_group2 = KaplanMeierFitter()
kmf_group2.fit(
    durations=data_2['allcause_follow_time'], 
    event_observed=data_2['allcause_event'], 
    label='UEBMI'
)

# 1 生存曲线
data_3 = df[df['医疗付款方式'] == 'SP']
kmf_group3 = KaplanMeierFitter()
kmf_group3.fit(
    durations=data_3['allcause_follow_time'], 
    event_observed=data_3['allcause_event'], 
    label='Self-pay'
)

# 2 生存曲线
data_4 = df[df['医疗付款方式'] == 'COI']
kmf_group4 = KaplanMeierFitter()
kmf_group4.fit(
    durations=data_4['allcause_follow_time'], 
    event_observed=data_4['allcause_event'], 
    label='commercial/other'
)

# 2 生存曲线
data_5 = df[df['医疗付款方式'] == 'GS']
kmf_group5 = KaplanMeierFitter()
kmf_group5.fit(
    durations=data_5['allcause_follow_time'], 
    event_observed=data_5['allcause_event'], 
    label='government-funded'
)

# # 绘制美化生存曲线
# plt.figure(figsize=(5,8))

ax = kmf_group1.plot(ci_show=True, linewidth=2, linestyle='--', color='#FBF165', legend=False)
kmf_group2.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#BB4EA2', legend=False)
kmf_group3.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#8083B9', legend=False)
kmf_group4.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#B48581', legend=False)
kmf_group5.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#88C6E1', legend=False)

add_at_risk_counts(kmf_group1, kmf_group2, kmf_group3, kmf_group4, kmf_group5, ax=ax) 

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(left=False, bottom=False)  # 去掉刻度

for ytick in ax.get_yticks():
    if ytick > 1 or ytick <= 0.61:
        continue
    ax.axhline(y=ytick, color="lightgray", linewidth=0.8, linestyle="--", zorder=0)



ax.set_xlabel("Time (months)", fontsize=14)
ax.set_ylabel("Survival Probability for allcause mortality")


plt.tight_layout()

plt.savefig('File/Figure/Part3/main/allcause_km.pdf')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# 0 生存曲线
data_1 = df[df['医疗付款方式'] == 'URRBMI']
kmf_group1 = KaplanMeierFitter()
kmf_group1.fit(
    durations=data_1['spe_follow_time'], 
    event_observed=data_1['spe_event'], 
    label='URBMI'
)

# 1 生存曲线
data_2 = df[df['医疗付款方式'] == 'UEBMI']
kmf_group2 = KaplanMeierFitter()
kmf_group2.fit(
    durations=data_2['spe_follow_time'], 
    event_observed=data_2['spe_event'], 
    label='UEBMI'
)

# 1 生存曲线
data_3 = df[df['医疗付款方式'] == 'SP']
kmf_group3 = KaplanMeierFitter()
kmf_group3.fit(
    durations=data_3['spe_follow_time'], 
    event_observed=data_3['spe_event'], 
    label='Self-pay'
)

# 2 生存曲线
data_4 = df[df['医疗付款方式'] == 'COI']
kmf_group4 = KaplanMeierFitter()
kmf_group4.fit(
    durations=data_4['spe_follow_time'], 
    event_observed=data_4['spe_event'], 
    label='commercial/other'
)

# 2 生存曲线
data_5 = df[df['医疗付款方式'] == 'GS']
kmf_group5 = KaplanMeierFitter()
kmf_group5.fit(
    durations=data_5['spe_follow_time'], 
    event_observed=data_5['spe_event'], 
    label='government-funded'
)

# # 绘制美化生存曲线
# plt.figure(figsize=(5,8))

ax = kmf_group1.plot(ci_show=True, linewidth=2, linestyle='--', color='#FBF165', legend=False)
kmf_group2.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#BB4EA2', legend=False)
kmf_group3.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#8083B9', legend=False)
kmf_group4.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#B48581', legend=False)
kmf_group5.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#88C6E1', legend=False)

add_at_risk_counts(kmf_group1, kmf_group2, kmf_group3, kmf_group4, kmf_group5, ax=ax) 

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(left=False, bottom=False)  # 去掉刻度

for ytick in ax.get_yticks():
    if ytick > 1 or ytick <= 0.72:
        continue
    ax.axhline(y=ytick, color="lightgray", linewidth=0.8, linestyle="--", zorder=0)



ax.set_xlabel("Time (months)", fontsize=14)
ax.set_ylim(0.72, 1)
# ax.set_ylabel("Survival Probability for cardio-cerebrovascular mortality")


plt.tight_layout()

plt.savefig('File/Figure/part3/main/spe_km.pdf')
# plt.show()

## Sensitivity samepayment

In [ ]:
import msoffcrypto
import io
import os

In [ ]:
icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

In [ ]:
county_list = [
    '范县', '方城县', '获嘉县',
    '郏县', '卢氏县', '杞县',
    '汤阴县', '武陟县', '夏邑县',
    '叶县'
]

unchanged_id = pay_change[pay_change['付款方式种类数'] == 1]['身份证号']
df_unchanged = df[df['身份证号'].isin(unchanged_id)].copy()

df_unchanged = df_unchanged[df_unchanged['区县'].isin(county_list)]

df_unchanged = df_unchanged.sort_values(by=['身份证号', '出生日期'], ascending=[True, True])
df_unchanged = df_unchanged.drop_duplicates(subset=['身份证号', '出生日期'], keep='first')
df_unchanged['身份证号'] = df_unchanged['身份证号'].apply(lambda x: x[-4:] if pd.notna(x) else x)


df_unchanged = df_unchanged.loc[:, ['姓名', '身份证号', '性别', '医疗付款方式','年龄', '区县', '入院时间'] + icd_cols]

print('没有改变过的医疗支付方式的记录有', df_unchanged.shape[0], '条', '\n' ,'共有', len(df_unchanged['身份证号'].unique()),'人')

In [ ]:
df_unchanged['key'] = df_unchanged['姓名'] + df_unchanged['身份证号']

In [ ]:
df_unchanged.shape

In [ ]:
payment_nunique = df_unchanged.groupby('key')['医疗付款方式'].transform('nunique')
df_same_payment = df_unchanged[payment_nunique == 1].copy()

In [ ]:
# 合并死亡数据
outcomes_list = []   
death_path = r"E:\DataBase\全省his系统\死亡数据"
password = os.environ.get('Death_code')

for i in county_list:
    # print(i)
    outcome_path = os.path.join(death_path, i, '死亡状态登记报告.xlsx')
    with open(outcome_path, 'rb') as f:
        office_file = msoffcrypto.OfficeFile(f)
        office_file.load_key(password=password)
        
        decrypted = io.BytesIO()
        office_file.decrypt(decrypted)  # 解密到内存

    death_df = pd.read_excel(decrypted, sheet_name=0)
    death_df['身份证号'] = death_df['身份证件号码'].apply(lambda x: x[-4:] if pd.notna(x) else x)

    outcomes_list.append(death_df.loc[:, ['姓名','身份证号', '死亡日期', '根本死因']].copy())
    
    death_df = pd.concat(outcomes_list)

In [ ]:
# 合并
df_same_payment = pd.merge(df_same_payment, death_df, on=['姓名','身份证号'], how='left')

In [ ]:
# 清洗日期格式
def clean_date_column(df, colname):
    """
    清洗日期列，统一格式为 YYYY-MM-DD
    - 支持斜杠/横杠
    - 自动补零（单个数字前补0）
    - 去除时分秒
    - 解析失败的值转换为 NaT
    """
    # 1. 转为字符串并清理
    df[colname] = (
        df[colname]
        .astype(str)
        .str.strip()
        .str.replace('/', '-', regex=False)                     # 统一分隔符
        .str.replace(r'(\D)(\d)(?=\D)', r'\g<1>0\2', regex=True)  # 单位数补零
        .str.slice(0, 10)                                       # 保留前10位 (YYYY-MM-DD)
    )

    # 2. 转为日期
    df[colname] = pd.to_datetime(df[colname], errors='coerce').dt.normalize()


    return df  

df_same_payment = clean_date_column(df_same_payment, '入院时间')
df_same_payment = clean_date_column(df_same_payment, '死亡日期')

In [ ]:
# 规范日期格式
def preprocess(df, start_time, end_time):
    """
    处理数据
    """

    # 入院时间
    df = df[(df['入院时间'] < pd.to_datetime(end_time)) & (df['入院时间'] > pd.to_datetime(start_time))]
    # 死亡日期
    df = df[(df['死亡日期'] < pd.to_datetime(end_time)) | pd.isna(df['死亡日期'])]
    # 入院时间-死亡日期
    df = df[(df['死亡日期'] > df['入院时间']) | pd.isna(df['死亡日期'])]

    # 去重
    df = df.drop_duplicates(subset=['姓名', '身份证号'], keep='first')

    # 重置索引
    df = df.reset_index(drop=True)

    return df

df_same_payment = preprocess(df_same_payment, start_time='2019-01-01', end_time='2024-12-31')

In [ ]:
icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

df_same_payment['heart'] = df_same_payment[icd_cols].isin(heart).any(axis=1).astype(int)
df_same_payment['brain'] = df_same_payment[icd_cols].isin(brain).any(axis=1).astype(int)
df_same_payment['kidney'] = df_same_payment[icd_cols].isin(kidney).any(axis=1).astype(int)

In [ ]:
# 生成follow_time和event
def creat_km_data(df, end_time, regex = False, time_col='follow_time', event_col='event'):
    """
    生成生存数据
    """
    import re

    # 随访时间
    end_time = pd.to_datetime(end_time)
    df['结束时间'] = df['死亡日期'].fillna(end_time)
    df[time_col] = (df['结束时间'] - df['入院时间']).dt.days / 30

    # 随访结局
    df[event_col] = 0
    if regex:
        df['status'] = 0

        mask = df['根本死因'].str.contains(regex, na=False, regex=True)
        df.loc[mask, event_col] = 1

        df.loc[mask, 'status'] = 1
        df.loc[(~mask) & (pd.notna(df['死亡日期'])), 'status'] = 2
    else:
        df.loc[pd.notna(df['死亡日期']), event_col] = 1

    return df

df_same_payment = creat_km_data(df_same_payment, end_time='2024-12-31', time_col='allcause_follow_time', event_col='allcause_event')
df_same_payment = creat_km_data(df_same_payment, end_time='2024-12-31', regex='.*心.*|.*脑.*',time_col='spe_follow_time', event_col='spe_event')
# df = creat_km_data(df, end_time='2024-12-31', regex='.*心.*',time_col='heart_follow_time', event_col='heart_event')
# df = creat_km_data(df, end_time='2024-12-31', regex='.*脑.*',time_col='brain_follow_time', event_col='brain_event')

In [ ]:
# df_same_payment.to_parquet('File/Part Data/part3-sensitivity-unchanged.parquet')
# df_same_payment = pd.read_parquet('File/Part Data/part3-sensitivity-unchanged.parquet')

In [ ]:
df_same_payment.shape

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# 0 生存曲线
data_1 = df_same_payment[df_same_payment['医疗付款方式'] == 'URRBMI']
kmf_group1 = KaplanMeierFitter()
kmf_group1.fit(
    durations=data_1['allcause_follow_time'], 
    event_observed=data_1['allcause_event'], 
    label='URBMI'
)

# 1 生存曲线
data_2 = df_same_payment[df_same_payment['医疗付款方式'] == 'UEBMI']
kmf_group2 = KaplanMeierFitter()
kmf_group2.fit(
    durations=data_2['allcause_follow_time'], 
    event_observed=data_2['allcause_event'], 
    label='UEBMI'
)

# 1 生存曲线
data_3 = df_same_payment[df_same_payment['医疗付款方式'] == 'SP']
kmf_group3 = KaplanMeierFitter()
kmf_group3.fit(
    durations=data_3['allcause_follow_time'], 
    event_observed=data_3['allcause_event'], 
    label='Self-pay'
)

# 2 生存曲线
data_4 = df_same_payment[df_same_payment['医疗付款方式'] == 'COI']
kmf_group4 = KaplanMeierFitter()
kmf_group4.fit(
    durations=data_4['allcause_follow_time'], 
    event_observed=data_4['allcause_event'], 
    label='commercial/other'
)

# 2 生存曲线
data_5 = df_same_payment[df_same_payment['医疗付款方式'] == 'GS']
kmf_group5 = KaplanMeierFitter()
kmf_group5.fit(
    durations=data_5['allcause_follow_time'], 
    event_observed=data_5['allcause_event'], 
    label='government-funded'
)

# # 绘制美化生存曲线
# plt.figure(figsize=(5,8))

ax = kmf_group1.plot(ci_show=True, linewidth=2, linestyle='--', color='#FBF165', legend=False)
kmf_group2.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#BB4EA2', legend=False)
kmf_group3.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#8083B9', legend=False)
kmf_group4.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#B48581', legend=False)
kmf_group5.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#88C6E1', legend=False)

add_at_risk_counts(kmf_group1, kmf_group2, kmf_group3, kmf_group4, kmf_group5, ax=ax) 

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(left=False, bottom=False)  # 去掉刻度

for ytick in ax.get_yticks():
    if ytick > 1 or ytick <= 0.61:
        continue
    ax.axhline(y=ytick, color="lightgray", linewidth=0.8, linestyle="--", zorder=0)



ax.set_xlabel("Time (months)", fontsize=14)
ax.set_ylabel("Survival Probability for allcause mortality")


plt.tight_layout()

plt.savefig('File/Figure/part3/sensitivity-unchanged.pdf')

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# 0 生存曲线
data_1 = df_same_payment[df_same_payment['医疗付款方式'] == 'URRBMI']
kmf_group1 = KaplanMeierFitter()
kmf_group1.fit(
    durations=data_1['spe_follow_time'], 
    event_observed=data_1['spe_event'], 
    label='URBMI'
)

# 1 生存曲线
data_2 = df_same_payment[df_same_payment['医疗付款方式'] == 'UEBMI']
kmf_group2 = KaplanMeierFitter()
kmf_group2.fit(
    durations=data_2['spe_follow_time'], 
    event_observed=data_2['spe_event'], 
    label='UEBMI'
)

# 1 生存曲线
data_3 = df_same_payment[df_same_payment['医疗付款方式'] == 'SP']
kmf_group3 = KaplanMeierFitter()
kmf_group3.fit(
    durations=data_3['spe_follow_time'], 
    event_observed=data_3['spe_event'], 
    label='Self-pay'
)

# 2 生存曲线
data_4 = df_same_payment[df_same_payment['医疗付款方式'] == 'COI']
kmf_group4 = KaplanMeierFitter()
kmf_group4.fit(
    durations=data_4['spe_follow_time'], 
    event_observed=data_4['spe_event'], 
    label='commercial/other'
)

# 2 生存曲线
data_5 = df_same_payment[df_same_payment['医疗付款方式'] == 'GS']
kmf_group5 = KaplanMeierFitter()
kmf_group5.fit(
    durations=data_5['spe_follow_time'], 
    event_observed=data_5['spe_event'], 
    label='government-funded'
)

# # 绘制美化生存曲线
# plt.figure(figsize=(5,8))

ax = kmf_group1.plot(ci_show=True, linewidth=2, linestyle='--', color='#FBF165', legend=False)
kmf_group2.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#BB4EA2', legend=False)
kmf_group3.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#8083B9', legend=False)
kmf_group4.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#B48581', legend=False)
kmf_group5.plot(ax=ax, ci_show=True, linewidth=2, linestyle='--', color='#88C6E1', legend=False)

add_at_risk_counts(kmf_group1, kmf_group2, kmf_group3, kmf_group4, kmf_group5, ax=ax) 

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(left=False, bottom=False)  # 去掉刻度

for ytick in ax.get_yticks():
    if ytick > 1 or ytick <= 0.72:
        continue
    ax.axhline(y=ytick, color="lightgray", linewidth=0.8, linestyle="--", zorder=0)



ax.set_xlabel("Time (months)", fontsize=14)
ax.set_ylim(0.72, 1)
# ax.set_ylabel("Survival Probability for cardio-cerebrovascular mortality")


plt.tight_layout()

plt.savefig('File/Figure/part3/sensitivity-unchanged/spe_km.pdf')
# plt.show()

# 基本信息统计

## Part1

In [ ]:
import pandas as pd
df = pd.read_parquet('File/hpconcat.parquet')
df = df[~df[['身份证号', '医疗付款方式', '入院时间', '性别', '年龄', '住院费', '自付金额']].isna().any(axis=1)]

In [ ]:
order = ['URRBMI', 'UEBMI', 'COI', 'SP', 'GS']

In [ ]:
df['医疗付款方式'].value_counts().sum()

In [ ]:
df['年龄'].mean(), df['年龄'].std()
df.groupby('医疗付款方式').agg(
    年龄_mean = ('年龄' , 'mean'),
    年龄_std = ('年龄' , 'std'),
).round(2).T[order]

In [ ]:
df['性别'] = df['性别'].astype(float)

In [ ]:
df['性别'].value_counts()

In [ ]:
# -- 性别 --
order = ['URRBMI', 'UEBMI', 'COI', 'SP', 'GS']
# 整体
# df['性别'].value_counts()
((df['性别'].value_counts() / df['性别'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['性别'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
df['住院费'].median(), df['住院费'].quantile(0.75)-df['住院费'].quantile(0.25)

mid = pd.DataFrame(df.groupby('医疗付款方式')['住院费'].median()).round(2)
std = pd.DataFrame(df.groupby('医疗付款方式')['住院费'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

In [ ]:
df_re = df[df['自付金额'] > 0]
df_re['自付金额'].median(), df_re['自付金额'].quantile(0.75)-df_re['自付金额'].quantile(0.25)

mid = pd.DataFrame(df_re.groupby('医疗付款方式')['自付金额'].median()).round(2)
std = pd.DataFrame(df_re.groupby('医疗付款方式')['自付金额'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

df_re['自付金额'].median(), df_re['自付金额'].quantile(0.75) - df_re['自付金额'].quantile(0.25)

In [ ]:
df['住院时长'] = (pd.to_datetime(df['出院时间']) - pd.to_datetime(df['入院时间'])).dt.days

In [ ]:
df['住院时长'].median(), df['住院时长'].quantile(0.75)-df['住院时长'].quantile(0.25)

mid = pd.DataFrame(df.groupby('医疗付款方式')['住院时长'].median()).round(2)
std = pd.DataFrame(df.groupby('医疗付款方式')['住院时长'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

In [ ]:
# 并发症确定

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df['hypertension'] = df[icd_cols].isin(hp).any(axis=1).astype(int)
df['heart'] = df[icd_cols].isin(heart).any(axis=1).astype(int)
df['brain'] = df[icd_cols].isin(brain).any(axis=1).astype(int)
df['kidney'] = df[icd_cols].isin(kidney).any(axis=1).astype(int)

In [ ]:
# -- heart --

# 整体
df['heart'].value_counts()
((df['heart'].value_counts() / df['heart'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['heart'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
# -- brain --

# 整体
df['brain'].value_counts()
((df['brain'].value_counts() / df['brain'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['brain'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
# -- kidney --

# 整体
df['kidney'].value_counts()
((df['kidney'].value_counts() / df['kidney'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['kidney'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

## Part3

In [ ]:
import pandas as pd
df = pd.read_parquet('File/Part Data/part3-main.parquet')
df = df[~df[['身份证号', '医疗付款方式', '入院时间', '性别', '年龄']].isna().any(axis=1)]

In [ ]:
order = ['URRBMI', 'UEBMI', 'COI', 'SP', 'GS']

In [ ]:
df['医疗付款方式'].value_counts()

In [ ]:
df['年龄'].mean(), df['年龄'].std()
df.groupby('医疗付款方式').agg(
    年龄_mean = ('年龄' , 'mean'),
    年龄_std = ('年龄' , 'std'),
).round(2).T[order]

In [ ]:
df['性别'] = df['性别'].astype(float)

In [ ]:
df['性别'].value_counts()

In [ ]:
# -- 性别 --
order = ['URRBMI', 'UEBMI', 'COI', 'SP', 'GS']
# 整体
# df['性别'].value_counts()
((df['性别'].value_counts() / df['性别'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['性别'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
df['住院费'].median(), df['住院费'].quantile(0.75)-df['住院费'].quantile(0.25)

mid = pd.DataFrame(df.groupby('医疗付款方式')['住院费'].median()).round(2)
std = pd.DataFrame(df.groupby('医疗付款方式')['住院费'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

In [ ]:
df_re = df[df['自付金额'] > 0]
df_re['自付金额'].median(), df_re['自付金额'].quantile(0.75)-df_re['自付金额'].quantile(0.25)

mid = pd.DataFrame(df_re.groupby('医疗付款方式')['自付金额'].median()).round(2)
std = pd.DataFrame(df_re.groupby('医疗付款方式')['自付金额'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

df_re['自付金额'].median(), df_re['自付金额'].quantile(0.75) - df_re['自付金额'].quantile(0.25)

In [ ]:
df['住院时长'] = (pd.to_datetime(df['出院时间']) - pd.to_datetime(df['入院时间'])).dt.days

In [ ]:
df['住院时长'].median(), df['住院时长'].quantile(0.75)-df['住院时长'].quantile(0.25)

mid = pd.DataFrame(df.groupby('医疗付款方式')['住院时长'].median()).round(2)
std = pd.DataFrame(df.groupby('医疗付款方式')['住院时长'].std()).round(2)
(mid.astype(str) + " (" + std.astype(str) + ")").T[order]

In [ ]:
# 并发症确定

# 高血压    I10-I15, 
# 心血管    I20-I25, I26-I28, I30-I52
# 脑血管    I60-I69
# 肾        N04, N18
hp = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']
heart = [
    'I20', 'I21', 'I22', 'I23', 'I24', 'I25', 
    'I26', 'I27', 'I28', 
    'I30', 'I31', 'I32', 'I33', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39', 'I40', 'I41', 'I42', 'I43', 'I44', 'I45', 'I46', 'I47', 'I48', 'I49', 'I50', 'I51', 'I52'
    ]
brain = ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69']
kidney = ['N04', 'N18']

icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

df['hypertension'] = df[icd_cols].isin(hp).any(axis=1).astype(int)
df['heart'] = df[icd_cols].isin(heart).any(axis=1).astype(int)
df['brain'] = df[icd_cols].isin(brain).any(axis=1).astype(int)
df['kidney'] = df[icd_cols].isin(kidney).any(axis=1).astype(int)

In [ ]:
# -- heart --

# 整体
df['heart'].value_counts()
((df['heart'].value_counts() / df['heart'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['heart'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
# -- brain --

# 整体
df['brain'].value_counts()
((df['brain'].value_counts() / df['brain'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['brain'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]

In [ ]:
# -- kidney --

# 整体
df['kidney'].value_counts()
((df['kidney'].value_counts() / df['kidney'].value_counts().sum())*100).round(2)

# 计数
cnt = pd.crosstab(df['医疗付款方式'], df['kidney'])
# 占比
pct = (cnt.div(cnt.sum(axis=1), axis=0).mul(100).round(2))
# 如果想把人数与占比合在一起显示：
out = cnt.astype(str) + ' (' + pct.round(2).astype(str) + ')'
out.T[order]